# IEEE 118-bus DP3Ph and EMT3Ph with SSN network components

This notebook builds the same IEEE 118-bus MATPOWER case in Python for DP3Ph and EMT3Ph, using the existing SSN PiLine, transformer, and impedance-load classes with the standard fourth-order synchronous generator model. A 10% resistive load step is applied at the largest load bus at 5 seconds. Both simulations run for 30 seconds, followed by voltage-magnitude plots in per unit. The SSN component bindings were added to dpsimpy; rebuild the `dpsimpy` target if your local Python extension predates this notebook.

In [ ]:
from pathlib import Path
import importlib
import os, re, sys, tempfile, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import savemat

_here = Path.cwd().resolve()
REPO = next(
    p for p in [_here, *_here.parents] if (p / "python" / "src" / "dpsim").is_dir()
)
os.chdir(REPO)
sys.path.insert(0, str(REPO / "build"))
sys.path.insert(0, str(REPO / "python" / "src"))
import dpsimpy

DP3PH_SWITCH = dpsimpy.dp.ph3.Switch
EMT_PH3_SWITCH = dpsimpy.emt.ph3.Switch
from dpsim import matpower as _matpower

matpower = importlib.reload(_matpower)

CASE_FILE = REPO / "dpsim" / "examples" / "matpower" / "case118.m"
FREQUENCY = 60.0
BASE_MVA = 100.0

## Load the MATPOWER case and assign common SG4 data

MATPOWER supplies the network and power-flow data, but not dynamic machine parameters. The same reasonable 4th-order parameters are assigned to each synchronous machine for both domains.

In [ ]:
case_text = CASE_FILE.read_text()


def parse_matrix(name):
    match = re.search(rf"mpc\.{name}\s*=\s*\[(.*?)\];", case_text, re.S)
    if not match:
        raise ValueError(f"Missing MATPOWER table {name}")
    rows = []
    for line in match.group(1).splitlines():
        line = line.split("%", 1)[0].strip().rstrip(";")
        if line:
            rows.append(np.fromstring(line, sep=" "))
    return np.vstack(rows)


BUS, GEN, BRANCH = (parse_matrix(k) for k in ("bus", "gen", "branch"))
assert BUS.shape == (118, 13) and len(GEN) == 54 and len(BRANCH) == 186
DYN = np.array(
    [
        [
            g[0],
            4,
            max(g[6], g[8]),
            4.0,
            0.002,
            0.15,
            5.0,
            1.0,
            0.5,
            0.05,
            1.75,
            0.425,
            0.3,
            1.75,
            0.425,
            0.3,
        ]
        for g in GEN
    ],
    dtype=float,
)
TEMP = Path(tempfile.mkdtemp(prefix="ieee118_ssn_"))
STATIC_PATH, DYNAMIC_PATH = TEMP / "case118.mat", TEMP / "case118_dyn.mat"
savemat(
    STATIC_PATH,
    {"mpc": {"baseMVA": BASE_MVA, "bus": BUS, "gen": GEN, "branch": BRANCH}},
)
savemat(DYNAMIC_PATH, {"mpc": {"gen_dyn": DYN}})
print(f"IEEE118: {len(BUS)} buses, {len(GEN)} SG4 machines, {len(BRANCH)} branches")

## Common power flow

The power flow is solved once and used to initialize both dynamic-domain topologies.

In [ ]:
pf_reader = matpower.Reader(str(STATIC_PATH))
pf_reader.create_dpsim_objects(
    matpower.Domain.PF,
    FREQUENCY,
    log_level=dpsimpy.LogLevel.off,
    filter_out_of_service=True,
)
pf_reader.create_dpsim_topology()
pf_sim = dpsimpy.Simulation("IEEE118_SSN_PF")
pf_sim.set_system(pf_reader.system)
pf_sim.set_domain(dpsimpy.Domain.SP)
pf_sim.set_solver(dpsimpy.Solver.NRP)
pf_sim.set_solver_component_behaviour(dpsimpy.SolverBehaviour.Initialization)
pf_sim.do_init_from_nodes_and_terminals(False)
pf_sim.set_time_step(0.01)
pf_sim.set_final_time(0.01)
pf_sim.run()
print("Solved common PF for", len(pf_reader.dpsimpy_busses_dict), "buses")

## Build each dynamic case with SSN network components

The MATPOWER reader still supplies the same bus and branch parameters and the standard `SynchronGenerator4OrderVBR` machines. The small subclass redirects only lines, transformers, loads, and shunts to their SSN counterparts. The switched load step is a conventional switch and resistor attached at one bus.

In [ ]:
class SSNMatpowerReader(matpower.Reader):
    def create_dpsim_objects(self, domain, *args, **kwargs):
        # MATPOWER chooses classes by looking them up on the domain module.
        # Temporarily redirect just branch components; generators remain standard SG4.
        module = dpsimpy.dp.ph3 if domain == matpower.Domain.DP3PH else dpsimpy.emt.ph3
        original = module.PiLine, module.Transformer
        module.PiLine, module.Transformer = module.SSN_PiLine, module.SSN_Transformer
        try:
            return super().create_dpsim_objects(domain, *args, **kwargs)
        finally:
            module.PiLine, module.Transformer = original

    def _add_ssn_load(
        self,
        bus_index,
        load_name,
        active_power,
        reactive_power,
        nominal_voltage,
        shunt_g=0.0,
        shunt_b=0.0,
    ):
        if (
            abs(active_power) < 1e-12
            and abs(reactive_power) < 1e-12
            and shunt_g == 0
            and shunt_b == 0
        ):
            return
        node = self.dpsimpy_busses_dict[self.get_node_name(bus_index)]
        load = self.dpsimpy_components.SSN_ImpedanceLoad(load_name, self.log_level)
        load.set_parameters(
            active_power, reactive_power, nominal_voltage, shunt_g, shunt_b
        )
        self.dpsimpy_comp_dict[load_name] = [load, [node]]

    def map_energy_consumer_dp3ph(
        self, bus_index, load_name, active_power, reactive_power, base_voltage, vm_pu
    ):
        self._add_ssn_load(
            bus_index, load_name, active_power, reactive_power, base_voltage * vm_pu
        )

    def map_energy_consumer(
        self,
        index,
        bus_index,
        load_name=None,
        bus_type=matpower.dpsimpy.PowerflowBusType.PQ,
    ):
        if self.domain == matpower.Domain.EMT:
            active_power = self.mpc_bus_data.at[index, "Pd"] * 1e6
            reactive_power = self.mpc_bus_data.at[index, "Qd"] * 1e6
            if (
                bus_type != matpower.dpsimpy.PowerflowBusType.PQ
                and self.domain != matpower.Domain.PF
            ):
                if abs(active_power) < 1e-12 and abs(reactive_power) < 1e-12:
                    return
            if load_name is None:
                self.loads_count += 1
                load_name = f"load{self.loads_count}"
            nominal_voltage = (
                self.mpc_bus_data.at[index, "baseKV"]
                * 1e3
                * self.mpc_bus_data.at[index, "Vm"]
            )
            self._add_ssn_load(
                bus_index, load_name, active_power, reactive_power, nominal_voltage
            )
            return
        return super().map_energy_consumer(index, bus_index, load_name, bus_type)

    def map_shunt(self, index, bus_index):
        if self.domain in (matpower.Domain.DP3PH, matpower.Domain.EMT):
            bus_base_v = self.mpc_bus_data.at[index, "baseKV"] * 1e3
            gs = self.mpc_bus_data.at[index, "Gs"] * 1e6 / bus_base_v**2
            bs = self.mpc_bus_data.at[index, "Bs"] * 1e6 / bus_base_v**2
            if gs != 0 or bs != 0:
                self._add_ssn_load(
                    bus_index, f"Shunt_N{bus_index}", 0.0, 0.0, bus_base_v, gs, bs
                )
            return
        return super().map_shunt(index, bus_index)


PLOT_BUSES = [1, 10, 30, 69, 100, 118]
LOAD_ROW = int(np.argmax(BUS[:, 2]))
LOAD_BUS = int(BUS[LOAD_ROW, 0])
LOAD_STEP_MW = 0.10 * float(BUS[LOAD_ROW, 2])
LOAD_STEP_TIME = 5.0
GEN_BUS = int(GEN[np.argmax(GEN[:, 8]), 0])
print(
    f"Load step: +{LOAD_STEP_MW:.1f} MW resistive at bus {LOAD_BUS} from {LOAD_STEP_TIME:g}s"
)

## Run DP3Ph and EMT3Ph for 30 seconds

Both cases use the same power-flow initialization, 54 conventional 4th-order SG models, and SSN lines, transformers, loads, and shunts. The step adds a 10% resistive increase at the largest-load bus and remains connected for the rest of each 30-second run.

In [ ]:
def run_case(domain, name, time_step):
    is_dp = domain == matpower.Domain.DP3PH
    reader = SSNMatpowerReader(str(STATIC_PATH), mpc_dyn_file_path=str(DYNAMIC_PATH))
    reader.create_dpsim_objects(
        domain,
        FREQUENCY,
        log_level=dpsimpy.LogLevel.off,
        with_pss=False,
        with_avr=False,
        with_tg=False,
        filter_out_of_service=True,
        generator_model=matpower.GenModel.SG_4ORDER_VBR,
    )

    # Add a shunt resistor switched in at t=5 s. P = V_LL^2/R for a balanced three-phase resistor.
    node = reader.dpsimpy_busses_dict[f"N{LOAD_BUS}"]
    base_kv = float(BUS[LOAD_ROW, 9])
    nominal_ll = base_kv * 1e3 * float(BUS[LOAD_ROW, 7])
    step_resistance = nominal_ll**2 / (LOAD_STEP_MW * 1e6)
    phase_parameter = dpsimpy.Math.single_phase_parameter_to_three_phase
    switch_cls = DP3PH_SWITCH if is_dp else EMT_PH3_SWITCH
    switch = switch_cls("LoadStep", dpsimpy.LogLevel.off)
    switch.set_parameters(phase_parameter(1e9), phase_parameter(step_resistance), False)
    ground = dpsimpy.dp.SimNode.gnd if is_dp else dpsimpy.emt.SimNode.gnd
    switch.connect([ground, node])
    reader.dpsimpy_comp_dict["LoadStep"] = [switch, [ground, node]]

    reader.create_dpsim_topology()
    reader.system.init_with_powerflow(
        systemPF=pf_reader.system,
        domain=dpsimpy.Domain.DP if is_dp else dpsimpy.Domain.EMT,
    )
    logger = dpsimpy.Logger(name, True, 1 if is_dp else 2)
    for bus_id in PLOT_BUSES:
        logger.log_attribute(
            f"N{bus_id}.V", "v", reader.dpsimpy_busses_dict[f"N{bus_id}"]
        )
    logger.log_attribute(
        f"Gen_N{GEN_BUS}.w_r", "w_r", reader.dpsimpy_comp_dict[f"Gen_N{GEN_BUS}"][0]
    )

    sim = dpsimpy.Simulation(name)
    sim.set_system(reader.system)
    sim.set_domain(dpsimpy.Domain.DP if is_dp else dpsimpy.Domain.EMT)
    sim.set_solver(dpsimpy.Solver.MNA)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_system_matrix_recomputation(True)
    sim.set_time_step(time_step)
    sim.set_final_time(30.0)
    sim.add_logger(logger)
    sim.add_event(dpsimpy.event.SwitchEvent3Ph(LOAD_STEP_TIME, switch, True))
    start = time.perf_counter()
    sim.run()
    runtime = time.perf_counter() - start
    trace = pd.read_csv(REPO / "logs" / f"{name}.csv", skipinitialspace=True)
    return reader, trace, runtime


dp_reader, dp_trace, dp_runtime = run_case(
    matpower.Domain.DP3PH, "IEEE118_SSN_DP3Ph_LoadStep", 1e-3
)
emt_reader, emt_trace, emt_runtime = run_case(
    matpower.Domain.EMT, "IEEE118_SSN_EMT3Ph_LoadStep", 5e-4
)
print(
    f"DP3Ph: {len(dp_trace)} samples, {dp_runtime:.2f}s runtime; EMT3Ph: {len(emt_trace)} samples, {emt_runtime:.2f}s runtime"
)
print(
    f"All logged values finite: {np.isfinite(dp_trace.iloc[:,1:].to_numpy()).all() and np.isfinite(emt_trace.iloc[:,1:].to_numpy()).all()}"
)

## Voltage magnitude plots

DP voltage is plotted directly from its complex phasor. EMT voltage is reduced to a one-cycle RMS envelope before conversion to per unit, so the two curves can be compared on the same magnitude scale.

In [ ]:
def dp_voltage_pu(trace, bus_id):
    v = np.hypot(
        trace[f"N{bus_id}.V_1.re"].to_numpy(), trace[f"N{bus_id}.V_1.im"].to_numpy()
    )
    return v / (float(BUS[bus_id - 1, 9]) * 1e3 * np.sqrt(2 / 3))


def emt_voltage_pu(trace, bus_id):
    phases = [f"N{bus_id}.V_{phase}" for phase in range(3)]
    phase_rms = np.sqrt((trace[phases].to_numpy() ** 2).mean(axis=1))
    phase_base = float(BUS[bus_id - 1, 9]) * 1e3 / np.sqrt(3)
    samples_per_cycle = max(
        2, round(1.0 / FREQUENCY / float(np.median(np.diff(trace.time.to_numpy()))))
    )
    envelope = np.sqrt(
        pd.Series((phase_rms / phase_base) ** 2)
        .rolling(samples_per_cycle, min_periods=samples_per_cycle)
        .mean()
    )
    return envelope.to_numpy()


fig, axes = plt.subplots(len(PLOT_BUSES), 1, figsize=(12, 13), sharex=True)
for ax, bus_id in zip(axes, PLOT_BUSES):
    ax.plot(dp_trace.time, dp_voltage_pu(dp_trace, bus_id), label="DP3Ph", lw=0.9)
    ax.plot(
        emt_trace.time,
        emt_voltage_pu(emt_trace, bus_id),
        label="EMT3Ph (one-cycle RMS)",
        lw=0.9,
    )
    ax.axvline(LOAD_STEP_TIME, color="black", ls="--", lw=0.8)
    ax.set_ylabel(f"Bus {bus_id}\n(pu)")
    ax.grid(True, alpha=0.25)
axes[0].legend(ncol=2)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("IEEE118 SSN network: bus voltage magnitudes with a 10% load step")
fig.tight_layout()
plt.show()

In [ ]:
print(f"Load step: +{LOAD_STEP_MW:.1f} MW at bus {LOAD_BUS}, t={LOAD_STEP_TIME:g}s")
print(f"Runtime ratio EMT/DP: {emt_runtime / dp_runtime:.2f}x")
for bus_id in PLOT_BUSES:
    dp_v = dp_voltage_pu(dp_trace, bus_id)
    emt_v = emt_voltage_pu(emt_trace, bus_id)
    print(
        f"Bus {bus_id:>3}: DP {np.nanmin(dp_v):.4f}–{np.nanmax(dp_v):.4f} pu; EMT {np.nanmin(emt_v):.4f}–{np.nanmax(emt_v):.4f} pu"
    )